# NLP4 - Text Processing Techniques: TF-IDF and LDA

In this session we will explore powerful techniques for understanding and analyzing text data. Two key concepts you'll learn are TF-IDF and LDA.

TF-IDF (Term Frequency-Inverse Document Frequency) is a method used to evaluate how important a word is in a document relative to a collection of documents. It helps filter out common words while highlighting those that are more meaningful in specific contexts.

LDA (Latent Dirichlet Allocation) is a topic modeling technique. It helps identify underlying topics in a set of documents by grouping words that frequently appear together.

These tools will give you insights into patterns in text data, opening doors to advanced text analysis!

---

## Install Libraries

In [ ]:
#first lets install datasets library
!pip install datasets
!python -m spacy download pt_core_news_lg

## Imports

In [2]:
#dataset library
from datasets import load_dataset
#load dataset
dataset = load_dataset("tclopess/sinopsys_movies_portuguese")
#convert it to pandas and slice the first 3000 data points
df_sinop = dataset['train'].to_pandas()[:3000]


#NLP tool box nltk
import nltk
from nltk.corpus import stopwords
#getting stop words
nltk.download('stopwords')
stop = list(set(stopwords.words('portuguese')))
print(stop)

#string library
import string
#get list of punctuations
pontuacoes = string.punctuation
print(pontuacoes)

#NLP toolbox spacy
import spacy
#load portuguese module large
nlp = spacy.load("pt_core_news_lg")

#other python support libraries and methods
import itertools
from collections import Counter
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#dataframes library
import pandas as pd

#LDA library
import gensim
import gensim.corpora as corpora
from gensim.models.ldamodel import LdaModel

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/645 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/625k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17947 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3168 [00:00<?, ? examples/s]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


['eu', 'houvera', 'tem', 'elas', 'tivemos', 'esteja', 'fui', 'não', 'fossem', 'me', 'de', 'houverão', 'haver', 'havemos', 'temos', 'te', 'sua', 'terão', 'sejamos', 'seremos', 'ele', 'uma', 'estiveram', 'pelos', 'seu', 'meu', 'dele', 'estava', 'estivéssemos', 'tivera', 'esses', 'houveriam', 'lhes', 'você', 'tenha', 'nas', 'muito', 'qual', 'aqueles', 'está', 'minha', 'ela', 'hajam', 'hajamos', 'minhas', 'e', 'mesmo', 'também', 'fôssemos', 'num', 'lhe', 'numa', 'pelo', 'houvéssemos', 'forem', 'teremos', 'eles', 'fomos', 'houvesse', 'o', 'no', 'os', 'do', 'tenham', 'teríamos', 'terei', 'teu', 'terá', 'sejam', 'teriam', 'tiver', 'houverei', 'mais', 'houveríamos', 'seriam', 'estamos', 'às', 'estão', 'nosso', 'tua', 'delas', 'seria', 'estejamos', 'há', 'estiver', 'ao', 'houver', 'éramos', 'até', 'seja', 'dos', 'houverem', 'foram', 'estávamos', 'por', 'sou', 'hão', 'eram', 'fosse', 'vocês', 'essa', 'estou', 'esta', 'isto', 'aquele', 'um', 'nossos', 'houve', 'ou', 'tuas', 'formos', 'fora', 'fôr

## Term Frequency - Inverse Document Frequency (TF-IDF)




The **TF-IDF** (Term Frequency - Inverse Document Frequency) model is an improvement over the Bag of Words. It not only takes into account the frequency of words in a document but also considers how important a word is in the entire corpus. The idea is that words that appear frequently in a document but rarely in the rest of the corpus are more meaningful for that document. TF-IDF assigns higher weights to such terms, thus reducing the impact of common words (e.g., "the", "and").

- **Term Frequency (TF)**: Measures how often a word appears in a document.
- **Inverse Document Frequency (IDF)**: Reduces the weight of commonly occurring words across multiple documents.

TF-IDF helps prioritize terms that are more informative for distinguishing between documents.

### Practicing


1 - Using the concepts from the previous class, create a function that takes a string as a parameter and returns a list of pre-processed tokens. The tokens should be lowercase, lemmas, and must not be punctuation or stopwords.

In [9]:
#funcao de preprocessing com lemmas
def preprocessing(text):
  return [x.lemma_.lower() for x in nlp(text) if x.text.lower() not in stop and x.text not in pontuacoes]


2 - Using the function you created in the previous exercise, preprocess all the synopsis texts contained in the dataframe.

In [10]:
preprocessed_docs = [preprocessing(x) for x in df_sinop['sinopse'].to_list()]

In [11]:
preprocessed_docs[0]

['história',
 'primeiro',
 'grande',
 'batalha',
 'fase',
 'americano',
 'guerra',
 'vietnã',
 'soldado',
 'ambos',
 'lado',
 'travar']

3 - Create a dataframe containing the tf values for all tokens in the documents. Consider the function below:

$$
TF(t,d) = \frac{\text{Number of times the term } t \text{ appears in the document } d } {\text{Total number of terms in the document } d}
$$



In [12]:
full_list = Counter(list(itertools.chain(*preprocessed_docs)))
print(full_list.most_common())

[('vida', 522), ('ano', 369), ('jovem', 342), ('descobrir', 337), ('dois', 319), ('encontrar', 310), ('enquanto', 294), ('amigo', 277), ('dever', 267), ('poder', 265), ('homem', 264), ('fazer', 260), ('história', 256), ('família', 251), ('pai', 243), ('tornar', 241), ('todo', 240), ('novo', 236), ('cidade', 235), ('começar', 231), ('casa', 213), ('mundo', 211), ('tentar', 207), ('sobre', 198), ('filme', 194), ('grande', 187), ('filho', 185), ('levar', 183), ('guerra', 177), ('onde', 177), ('outro', 174), ('contra', 171), ('matar', 162), ('mulher', 162), ('lutar', 160), ('conhecer', 157), ('durante', 154), ('após', 152), ('grupo', 150), ('ficar', 149), ('assassino', 143), ('amor', 136), ('vez', 132), ('...', 131), ('próprio', 131), ('assassinato', 127), ('dia', 126), ('ter', 126), ('morte', 124), ('chamar', 123), ('terra', 122), ('policial', 121), ('filha', 121), ('tempo', 121), ('tudo', 120), ('mãe', 120), ('adolescente', 119), ('ver', 119), ('irmão', 119), ('três', 118), ('misterioso'

In [13]:
list_dict_tfs = []

for doc in preprocessed_docs:
  dict_doc = defaultdict(int)
  total_doc = len(doc)
  cont_doc = Counter(doc)
  for word in full_list:
    if word in cont_doc.keys():
        #faz uma soma
        dict_doc[word] = cont_doc[word]/total_doc
    else:
        dict_doc[word] = 0

  list_dict_tfs.append(dict_doc)

In [14]:
df_tf = pd.DataFrame(list_dict_tfs)

In [16]:
df_tf

,história,primeiro,grande,batalha,fase,americano,guerra,vietnã,soldado,ambos,...,finding,neverland,pan,j.m.,barrie,sylvia,wilder,lemmon,matthau,esgotado
0,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2996,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2997,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2998,0.028571,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.028571,0.057143,0.057143,0.028571,0.057143,0.028571,0.000000,0.000000,0.000000,0.000000


4 - Now consider the IDF formula below. Calculate an IDF vector for all tokens in the corpus.

$$
IDF(t) = \log \left( \frac{\text{Number of documents in the corpus}}{\text{Number of documents where the term } t \text{ appears}} \right)
$$

In [24]:
#total of documents where t appears
#df(t) = occurrence of t in documents¶
# aqui desconsidero se o termo aparece mais de uma vez no mesmo documento
dic_dt = defaultdict(int)
for document in preprocessed_docs:
    #aqui retira os termos repetidos
    for token in set(document):
        dic_dt[token]+=1

# dic_dt

In [25]:
dic_dt['história']

232

In [26]:
#calcula o idf para cada termo
# o idf é um calculo do termo em relaçao ao set de documentos

import math

dic_idf = {}
for key in dic_dt:
    dic_idf[key]=math.log(len(preprocessed_docs)/((dic_dt[key])))


dic_idf['história']

2.559630195983937

5 - Analyze the TF and IDF separately. What would be their relationship with the corpus, with a specific document, or with a specific term?

In [27]:
#TF measures how often a term appears in a document relative to the total number of terms in that document. It indicates the importance of a term within that specific document.
#IDF assesses the importance of a term across the entire corpus (a collection of documents). It measures how unique or rare a term is. A term that appears in many documents will have a lower IDF score, while a term that appears in few documents will have a higher IDF score.

6 - Using the data structures you used to separately calculate the TF and IDF above, return the TF-IDF value for the token 'história' in document 45.

In [28]:
#tfs do termo para os documentos que ele aparece
token = 'história'
print(df_tf.loc[45,token]*dic_idf['história'])

0.15997688724899606


## Consine Similarity

In the BOW model, texts are represented as vectors that count the occurrence of words in each document, ignoring word order and focusing on frequency. The similarity between documents can be assessed using these vectors through metrics like **cosine similarity**. Cosine similarity measures the angle between two vectors, determining how similar the documents are based on the words they share, even if in different quantities. This allows for efficient comparison of text content using the vector representations created by BOW.


### Practicing

1 - Consider the vectors below. Which ones are most similar to each other?

In [ ]:
X = [0, 0, 0, 1, 1, 1]
Y = [1, 0, 0, 1, 1, 0]
Z = [0, 1, 0, 0, 0, 0]

2 - Answer the question above using the `cosine_similarity` function.

In [ ]:
X = np.array([0, 0, 0, 1, 1, 1]).reshape(1, 6)
Y = np.array([1, 0, 0, 1, 1, 0]).reshape(1, 6)
Z = np.array([0, 1, 0, 0, 0, 0]).reshape(1, 6)
print('X e Y', cosine_similarity(X, Y))
print('X e Z', cosine_similarity(X, Z))
print('Y e Z', cosine_similarity(Y, Z))

X e Y [[0.66666667]]
X e Z [[0.]]
Y e Z [[0.]]


2 -  Create a dataframe for the analyzed corpus, where each row represents a document and each column represents the unique tokens. Each row will therefore indicate how many times a particular token appears in a given document.

In [ ]:
#lista para criar um dataframe com os vetores BoW
list_dict_documents = []

#para cada documento
for document in preprocessed_docs:
    #cria-se um dicionado default
    cont_doc = Counter(document)
    dict_doc = defaultdict(int)
    for key in full_list.keys():
        if key in cont_doc.keys():
            #faz uma soma
            dict_doc[key] = cont_doc[key]
        else:
            dict_doc[key] = 0

    #adiciona ao final do looping a linha referente ao documento
    list_dict_documents.append(dict_doc)

In [ ]:
#criando o dataframe
df_cont = pd.DataFrame(list_dict_documents)
df_cont

,história,primeiro,grande,batalha,fase,americano,guerra,vietnã,soldado,ambos,...,finding,neverland,pan,j.m.,barrie,sylvia,wilder,lemmon,matthau,esgotado
0,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2998,1,0,0,0,0,0,0,0,0,0,...,1,2,2,1,2,1,0,0,0,0


3 - Consider the synopses below. Which of the 3 are most similar or discuss the same topic?

In [ ]:
df_sinop.loc[636,'sinopse']

'Quando a família de Frank Castle é assassinada por criminosos, ele trava uma guerra contra o crime como um assassino vigilante conhecido apenas como O Justiceiro.'

In [ ]:
df_sinop.loc[999,'sinopse']

'O mafioso e assassino de aluguel Jimmy Conlon tem uma noite para descobrir onde está sua lealdade: com seu filho distante, Mike, cuja vida está em perigo, ou seu melhor amigo de longa data, o chefe da máfia Shawn Maguire, que quer que Mike pague pela morte de seu próprio filho.'

In [ ]:
df_sinop.loc[14,'sinopse']

'Em julho de 1969, a corrida espacial terminou quando a Apollo 11 cumpriu o desafio do presidente Kennedy de “pousar um homem na Lua e trazê-lo de volta são e salvo à Terra”. Ninguém que testemunhou o pouso lunar jamais o esquecerá. O documentário de Al Reinert, For All Mankind, é a história dos vinte e quatro homens que viajaram para a lua, contada em suas palavras, em suas vozes, usando as imagens de suas experiências. Quarenta anos após o primeiro pouso na lua, continua sendo a obra de cinema mais radical e visualmente deslumbrante já feita sobre esse evento de abalar a terra.'

4 - Use the cosine similarity function to justify your answer.

In [ ]:
X = np.array(df_cont.loc[636]).reshape(1,12615)
Y = np.array(df_cont.loc[999]).reshape(1,12615)
# X = [[0, 0, 0], [1, 1, 1]]
# Y = [[1, 0, 0], [1, 1, 0]]
cosine_similarity(X, Y)

array([[0.04652421]])

In [ ]:
X = np.array(df_cont.loc[636]).reshape(1,12615)
Y = np.array(df_cont.loc[14]).reshape(1,12615)
# X = [[0, 0, 0], [1, 1, 1]]
# Y = [[1, 0, 0], [1, 1, 0]]
cosine_similarity(X, Y)

array([[0.]])

In [ ]:
X = np.array(df_cont.loc[999]).reshape(1,12615)
Y = np.array(df_cont.loc[14]).reshape(1,12615)
# X = [[0, 0, 0], [1, 1, 1]]
# Y = [[1, 0, 0], [1, 1, 0]]
cosine_similarity(X, Y)

array([[0.]])

## Topic Modelling and LDA

While TF-IDF is effective for identifying key terms, it doesn’t provide insight into the underlying topics within the text.

This is where **Topic Modeling** comes in. It’s a technique used to automatically uncover hidden topics in large collections of text. A widely-used topic modeling method is **Latent Dirichlet Allocation (LDA)**, which goes beyond word frequencies to model the distribution of topics across documents and the distribution of words within topics. LDA assumes that each document consists of multiple topics, and each topic consists of related words.


### Practicing

1 - Discuss the paper that originated LDA. Take notes below in order to understand what the model is and how a single document can be composed of multiple topics.

2 - For this study, we will use the `Gensim` library. The first step is to create a dictionary. Use `corpora.Dictionary()` to create a dictionary that we will use in the model. Understand what this dictionary is. Did we use the same preprocessing that we did for TF-IDF?

In [ ]:
# Create Dictionary
id2word = corpora.Dictionary(preprocessed_docs)

In [ ]:
id2word[0]

'ambos'

3 - As a second input, it is necessary to create the corpus for the `LdaModel()` function. Read the documentation and create a compatible corpus based on the preprocessing you have already done.

In [ ]:
# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in preprocessed_docs]

In [ ]:
corpus[0]

[(0, 1),
 (1, 1),
 (2, 1),
 (3, 1),
 (4, 1),
 (5, 1),
 (6, 1),
 (7, 1),
 (8, 1),
 (9, 1),
 (10, 1),
 (11, 1)]

4 - One of the most important steps for topic modeling algorithms is determining how many topics to use as input. Discuss how this decision should be made. For testing, use `num_topics=10`.

In [ ]:
# Set number of topics
num_topics = 10

5 - Finally, create a model from the objects created so far using the function `LdaModel()`

In [ ]:
# Build LDA model
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=num_topics,
    passes=20,
    random_state=100
    )

6 - Explain and discuss what the parameters `random_state` and `passes` refer to.

In [ ]:
# random_state: This is comparable to 'seed' in many packages and libraries. LDA is probabilistic, not deterministic. This means that, even were we to train models with the same parameters on the same corpus, our models might vary minutely each time. random_state helps mitigate this variation and thereby aid reproducibility.
# passes: The number of times the algorithm passes through the whole corpus. This is comparable to 'epochs' in other packages and libraries.

7 - LDA provides two main outputs, the loadings and the scores. What do they refer to?

In [ ]:
# Loadings: These refer to the weights or coefficients that indicate how much each word contributes to the topic.
# Scores: Represent the transformed values of the data in terms of the new axes and are used to classify the observations into different groups or classes.

8 - Use `lda_model.print_topics()` to access the tokens that contribute to each of the created topics (Loadings).

In [ ]:
# Print the keywords for each topic
#Researchers use both qualitative and quantitative methods to evaluate models
lda_model.print_topics(num_words=20)

#Topics are words with highest probability in topic and the numbers are the probabilities of words appearing in topic distribution.

[(0,
  '0.005*"ladrão" + 0.004*"jones" + 0.004*"bruxa" + 0.004*"força" + 0.004*"próprio" + 0.004*"banco" + 0.004*"enquanto" + 0.004*"coração" + 0.004*"tornar" + 0.003*"agora" + 0.003*"serviço" + 0.003*"lutar" + 0.003*"desconhecer" + 0.003*"tentar" + 0.003*"equipe" + 0.003*"destruir" + 0.003*"último" + 0.003*"inimigo" + 0.003*"sobre" + 0.003*"chamar"'),
 (1,
  '0.008*"guerra" + 0.005*"exército" + 0.005*"gangue" + 0.004*"filme" + 0.004*"durante" + 0.004*"mundial" + 0.004*"robert" + 0.004*"bandido" + 0.004*"oficial" + 0.004*"americano" + 0.003*"ano" + 0.003*"the" + 0.003*"dois" + 0.003*"romance" + 0.003*"campo" + 0.003*"mudança" + 0.003*"linha" + 0.003*"lucy" + 0.003*"quebrar" + 0.003*"chamar"'),
 (2,
  '0.008*"tornar" + 0.007*"enquanto" + 0.007*"vida" + 0.006*"novo" + 0.005*"ano" + 0.004*"encontrar" + 0.004*"história" + 0.004*"relacionamento" + 0.004*"dar" + 0.004*"york" + 0.003*"nova" + 0.003*"todo" + 0.003*"homem" + 0.003*"sobre" + 0.003*"poder" + 0.003*"dois" + 0.003*"velho" + 0.003*"

9 - Create a `for` loop to print each document and its distribution among topics. Use `lda_model.get_document_topics()`(Scores).

In [ ]:
# generate document-topic distributions
for i, doc in enumerate(corpus):
    doc_topics = lda_model.get_document_topics(doc)
    print(f"Document {i}: {doc_topics}")

Document 0: [(1, 0.2542863), (4, 0.6841163)]
Document 1: [(1, 0.3727837), (9, 0.59383684)]
Document 2: [(4, 0.8323124), (8, 0.14542103)]
Document 3: [(0, 0.11194908), (3, 0.41749018), (4, 0.42046648)]
Document 4: [(0, 0.13084902), (2, 0.33413678), (6, 0.14182718), (7, 0.36916292)]
Document 5: [(4, 0.067950524), (9, 0.89389926)]
Document 6: [(0, 0.09560691), (1, 0.028979938), (3, 0.5876315), (5, 0.11230724), (7, 0.16619892)]
Document 7: [(0, 0.14820151), (3, 0.70505464), (8, 0.10003187)]
Document 8: [(4, 0.15756814), (7, 0.8138252)]
Document 9: [(1, 0.3079705), (4, 0.28584442), (7, 0.35608253)]
Document 10: [(3, 0.30184272), (4, 0.6765081)]
Document 11: [(2, 0.56374234), (4, 0.39409056)]
Document 12: [(3, 0.20310168), (4, 0.32139966), (7, 0.2758296), (8, 0.17233744)]
Document 13: [(0, 0.3011899), (3, 0.37201503), (6, 0.30176213)]
Document 14: [(3, 0.031370737), (4, 0.28720987), (6, 0.42200932), (7, 0.24971616)]
Document 15: [(7, 0.96083146)]
Document 16: [(0, 0.16077344), (4, 0.12636837

10 - Discuss the score in terms of what score would be sufficient to determine whether a document belongs to a topic or not.

11 - EXTRA

Study the pyLDAvis library to create direct graphs from the gensim library related to the model you just created.

In [ ]:
!pip install pyLDAvis

In [ ]:
# for LDA evaluation
import pyLDAvis
import pyLDAvis.gensim_models as gensimvisualize

pyLDAvis.enable_notebook(local=True)

In [ ]:
dickens_visual = gensimvisualize.prepare(lda_model, corpus, id2word, mds='mmds',sort_topics=False)
pyLDAvis.display(dickens_visual)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
